In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
import enum
import pandas as pd

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
physical_devices

In [ ]:
num_points = 21
values_per_point = 3

model = Sequential()

model.add(Dense(512, activation='relu', input_shape=(num_points * values_per_point,), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.3))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(24, activation='sigmoid'))

In [ ]:
data = pd.read_csv("video_dataset_fhpq_no_rotate.csv")
data.head()

In [ ]:
# Podziel dane na funkcje wejściowe X i etykiety y
X = data.iloc[:, :63]  # wszystkie kolumny oprócz ostatnich
y = data.iloc[:, 63:]  # tylko ostatnia kolumna

# Podziel dane na zbiory treningowe i testowe
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)


# Skonwertuj DataFrame do numpy arrays
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values
X_val = X_val.values
y_val = y_val.values

In [ ]:

metrics = tf.keras.metrics.AUC(
    num_thresholds=200,
    curve='ROC',
    summation_method='interpolation',
    name=None,
    dtype=None,
    thresholds=None,
    multi_label=False,
    num_labels=None,
    label_weights=None,
    from_logits=False
)

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=[metrics])

In [ ]:
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32)

In [ ]:
loss = model.evaluate(X_test, y_test)

In [ ]:
model.save('models/video_dataset_fhpq_no_rotate_xyz_100_epoch.keras')